# Train HeatGeo from GitHub (Colab)

Notebook này luôn đồng bộ code mới nhất từ branch `nqd_mass_geom_loss` trước khi cài dependencies và train. Chọn một trong ba teacher–student pair, chỉnh `MASS_WEIGHT`, rồi dùng **Runtime → Run all**.

> Lưu ý: cell đồng bộ code dùng `git reset --hard` bên trong `/content/embedding-kd`, nên các chỉnh sửa tracked cục bộ trong bản clone Colab sẽ bị bỏ. Cache và output untracked không bị xóa. Pair Qwen3-4B → BERT-base cần GPU có VRAM lớn.

In [14]:
#@title 1. Cấu hình experiment
REPO_URL = "https://github.com/Savoxism/embedding-kd.git"
BRANCH = "nqd_mass_geom_loss"

PAIR_KEY = "qwen3_0_6b_to_minilmv2_h384" #@param 
# ["qwen3_0_6b_to_minilmv2_h384", "bge_m3_to_minilmv2_h768", "qwen3_4b_to_bert_base"]
MASS_WEIGHT = 0.5 #@param {type:"number"}
RUN_NAME = "mass_loss_run" #@param {type:"string"}

BATCH_SIZE = 32 #@param {type:"integer"}
EPOCHS = 5 #@param {type:"integer"}
LEARNING_RATE = 2e-5 #@param {type:"number"}
MAX_LENGTH = 256 #@param {type:"integer"}
SEED = 42 #@param {type:"integer"}
NUM_WORKERS = 2 #@param {type:"integer"}
TRAIN_DATA = "data/train_set/merged_3_data_5k_each.csv" #@param {type:"string"}

USE_WANDB = False #@param {type:"boolean"}
WANDB_PROJECT = "iclr-mdd-heatgeo" #@param {type:"string"}
WANDB_MODE = "online" #@param ["online", "offline", "disabled"]
FINAL_WEIGHTS_ONLY = True #@param {type:"boolean"}
REQUIRE_GPU = True #@param {type:"boolean"}

USE_GOOGLE_DRIVE = False #@param {type:"boolean"}
DRIVE_ROOT = "/content/drive/MyDrive/embedding-kd-runs" #@param {type:"string"}

# Thêm CLI flags nếu cần, ví dụ: --graph_k 100 --diffusion_quota 20
EXTRA_ARGS = "" #@param {type:"string"}

assert MASS_WEIGHT >= 0, "MASS_WEIGHT phải không âm"
assert BATCH_SIZE > 0 and EPOCHS > 0 and MAX_LENGTH > 0


In [15]:
# 2. Clone lần đầu; các lần Run all sau luôn fetch/reset/pull branch mới nhất
from pathlib import Path
import subprocess

REPO_DIR = Path("/content/embedding-kd")

def git(*args):
    command = ["git", "-C", str(REPO_DIR), *args]
    print("+", " ".join(command))
    subprocess.run(command, check=True)

if (REPO_DIR / ".git").is_dir():
    git("remote", "set-url", "origin", REPO_URL)
    # Bỏ thay đổi tracked trong clone Colab để luôn checkout được remote head.
    git("reset", "--hard")
    git("fetch", "--prune", "origin")
    git("checkout", "-B", BRANCH, f"origin/{BRANCH}")
    git("reset", "--hard", f"origin/{BRANCH}")
    git("pull", "--ff-only", "origin", BRANCH)
elif REPO_DIR.exists() and any(REPO_DIR.iterdir()):
    raise RuntimeError(f"{REPO_DIR} tồn tại nhưng không phải Git repository")
else:
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )

commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
print(f"Ready: {BRANCH}@{commit[:12]}")


+ git -C /content/embedding-kd remote set-url origin https://github.com/Savoxism/embedding-kd.git
+ git -C /content/embedding-kd reset --hard
+ git -C /content/embedding-kd fetch --prune origin
+ git -C /content/embedding-kd checkout -B nqd_mass_geom_loss origin/nqd_mass_geom_loss
+ git -C /content/embedding-kd reset --hard origin/nqd_mass_geom_loss
+ git -C /content/embedding-kd pull --ff-only origin nqd_mass_geom_loss
Ready: nqd_mass_geom_loss@297fe991215b


In [16]:
# 3. Cài/đồng bộ dependencies theo code vừa pull
import os
import sys

os.chdir(REPO_DIR)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")],
    check=True,
)
print("Dependencies are ready.")


Dependencies are ready.


In [17]:
# 4. Resolve model pair, GPU và nơi lưu artifacts
import torch

PAIR_CONFIGS = {
    "qwen3_0_6b_to_minilmv2_h384": {
        "teacher": "Qwen/Qwen3-Embedding-0.6B",
        "student": "nreimers/MiniLMv2-L6-H384-distilled-from-BERT-Base",
        "pooling": "last_token",
    },
    "bge_m3_to_minilmv2_h768": {
        "teacher": "BAAI/bge-m3",
        "student": "nreimers/MiniLMv2-L6-H768-distilled-from-BERT-Base",
        "pooling": "cls",
    },
    "qwen3_4b_to_bert_base": {
        "teacher": "Qwen/Qwen3-Embedding-4B",
        "student": "google-bert/bert-base-uncased",
        "pooling": "last_token",
    },
}
pair = PAIR_CONFIGS[PAIR_KEY]

if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError("Không tìm thấy CUDA GPU. Trong Colab chọn Runtime → Change runtime type → GPU.")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 2**30
    print(f"GPU: {props.name} ({vram_gb:.1f} GiB)")
    if PAIR_KEY == "qwen3_4b_to_bert_base" and vram_gb < 35:
        print("WARNING: Qwen3-4B ở cấu hình hiện tại có thể OOM trên GPU dưới khoảng 35 GiB.")

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    artifact_root = Path(DRIVE_ROOT)
else:
    artifact_root = REPO_DIR

cache_dir = artifact_root / "cache" / "heatgeo" / PAIR_KEY
log_dir = artifact_root / "logs" / "heatgeo" / PAIR_KEY
save_dir = artifact_root / "models" / "heatgeo" / PAIR_KEY / RUN_NAME
weights_dir = artifact_root / "models" / "heatgeo_weights" / PAIR_KEY / RUN_NAME
for path in (cache_dir, log_dir, save_dir, weights_dir):
    path.mkdir(parents=True, exist_ok=True)

print(f"Teacher: {pair['teacher']}")
print(f"Student: {pair['student']}")
print(f"Pooling: {pair['pooling']}")
print(f"Objective: L_rel + {MASS_WEIGHT} * L_mass")
print(f"Outputs: {save_dir}")


GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition (95.0 GiB)
Teacher: Qwen/Qwen3-Embedding-0.6B
Student: nreimers/MiniLMv2-L6-H384-distilled-from-BERT-Base
Pooling: last_token
Objective: L_rel + 0.5 * L_mass
Outputs: /content/embedding-kd/models/heatgeo/qwen3_0_6b_to_minilmv2_h384/mass_loss_run


In [18]:
# 5. Train HeatGeo
import shlex

command = [
    sys.executable, "main.py",
    "--method", "heatgeo",
    "--train_data", TRAIN_DATA,
    "--student_model", pair["student"],
    "--teacher_model", pair["teacher"],
    "--pooling_method", pair["pooling"],
    "--mass_weight", str(MASS_WEIGHT),
    "--batch_size", str(BATCH_SIZE),
    "--epochs", str(EPOCHS),
    "--lr", str(LEARNING_RATE),
    "--max_length", str(MAX_LENGTH),
    "--seed", str(SEED),
    "--num_workers", str(NUM_WORKERS),
    "--cache_path", str(cache_dir / "teacher_train.pt"),
    "--heatgeo_cache_path", str(cache_dir / "graph.pt"),
    "--heatgeo_log_dir", str(log_dir),
    "--save_dir", str(save_dir),
    "--weights_dir", str(weights_dir),
    "--wandb_project", WANDB_PROJECT,
    "--wandb_run_name", f"{PAIR_KEY}_{RUN_NAME}",
    "--wandb_mode", WANDB_MODE,
]
if FINAL_WEIGHTS_ONLY:
    command.append("--final_weights_only")
if not USE_WANDB:
    command.append("--no_wandb")
if EXTRA_ARGS.strip():
    command.extend(shlex.split(EXTRA_ARGS))

env = os.environ.copy()
env["TOKENIZERS_PARALLELISM"] = "false"
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("+", shlex.join(command))
subprocess.run(command, cwd=REPO_DIR, env=env, check=True)


+ /usr/bin/python3 main.py --method heatgeo --train_data data/train_set/merged_3_data_5k_each.csv --student_model nreimers/MiniLMv2-L6-H384-distilled-from-BERT-Base --teacher_model Qwen/Qwen3-Embedding-0.6B --pooling_method last_token --mass_weight 0.5 --batch_size 32 --epochs 5 --lr 2e-05 --max_length 256 --seed 42 --num_workers 2 --cache_path /content/embedding-kd/cache/heatgeo/qwen3_0_6b_to_minilmv2_h384/teacher_train.pt --heatgeo_cache_path /content/embedding-kd/cache/heatgeo/qwen3_0_6b_to_minilmv2_h384/graph.pt --heatgeo_log_dir /content/embedding-kd/logs/heatgeo/qwen3_0_6b_to_minilmv2_h384 --save_dir /content/embedding-kd/models/heatgeo/qwen3_0_6b_to_minilmv2_h384/mass_loss_run --weights_dir /content/embedding-kd/models/heatgeo_weights/qwen3_0_6b_to_minilmv2_h384/mass_loss_run --wandb_project iclr-mdd-heatgeo --wandb_run_name qwen3_0_6b_to_minilmv2_h384_mass_loss_run --wandb_mode online --final_weights_only --no_wandb


CompletedProcess(args=['/usr/bin/python3', 'main.py', '--method', 'heatgeo', '--train_data', 'data/train_set/merged_3_data_5k_each.csv', '--student_model', 'nreimers/MiniLMv2-L6-H384-distilled-from-BERT-Base', '--teacher_model', 'Qwen/Qwen3-Embedding-0.6B', '--pooling_method', 'last_token', '--mass_weight', '0.5', '--batch_size', '32', '--epochs', '5', '--lr', '2e-05', '--max_length', '256', '--seed', '42', '--num_workers', '2', '--cache_path', '/content/embedding-kd/cache/heatgeo/qwen3_0_6b_to_minilmv2_h384/teacher_train.pt', '--heatgeo_cache_path', '/content/embedding-kd/cache/heatgeo/qwen3_0_6b_to_minilmv2_h384/graph.pt', '--heatgeo_log_dir', '/content/embedding-kd/logs/heatgeo/qwen3_0_6b_to_minilmv2_h384', '--save_dir', '/content/embedding-kd/models/heatgeo/qwen3_0_6b_to_minilmv2_h384/mass_loss_run', '--weights_dir', '/content/embedding-kd/models/heatgeo_weights/qwen3_0_6b_to_minilmv2_h384/mass_loss_run', '--wandb_project', 'iclr-mdd-heatgeo', '--wandb_run_name', 'qwen3_0_6b_to_min

In [19]:
# 6. Xem artifacts và các metrics cuối
import json

print("Checkpoints:", save_dir)
print("Weights:", weights_dir)
metrics_path = save_dir / "metrics.jsonl"
if metrics_path.exists():
    records = [json.loads(line) for line in metrics_path.read_text().splitlines() if line.strip()]
    print(json.dumps(records[-1], indent=2, ensure_ascii=False) if records else "metrics.jsonl is empty")
else:
    print("Không tìm thấy metrics.jsonl")


Checkpoints: /content/embedding-kd/models/heatgeo/qwen3_0_6b_to_minilmv2_h384/mass_loss_run
Weights: /content/embedding-kd/models/heatgeo_weights/qwen3_0_6b_to_minilmv2_h384/mass_loss_run
{
  "method": "heatgeo",
  "seed": 42,
  "test": {
    "classification": {
      "data/test_set/banking77_test.csv": {
        "accuracy": 0.8735370611183355,
        "f1": 0.8738861385303659
      },
      "data/test_set/emotion_test.csv": {
        "accuracy": 0.7225579053373615,
        "f1": 0.641400416946333
      },
      "data/test_set/tweet_test.csv": {
        "accuracy": 0.7062937062937062,
        "f1": 0.710467853180924
      }
    },
    "pair": {
      "data/test_set/mrpc_test.csv": {
        "accuracy": 0.6921739130434783,
        "average_precision": 0.8163425859986241,
        "best_threshold": 0.8743718592964824,
        "f1": 0.5427963756140097,
        "precision": 0.6714468396054074,
        "recall": 0.5651179698506409
      },
      "data/test_set/scitail_test.csv": {
        "a